# TFBS: train and compare all models on the real FOSL2 data

**What this notebook does**
1. Loads the real dataset: ENCODE ChIP-seq for the transcription factor **FOSL2** in HepG2 liver cells, 101 bp sequences, about 39k training and 10k test rows, from the BERT-TFBS benchmark.
2. Trains every model on **exactly the same data**: CNN, CNN + BiLSTM, and CNN + BiLSTM + Transformer (our main model).
3. Uses **5-fold cross-validation**: each model is trained 5 times on different slices of the training data, so the result isn't luck. The official test set is never used for training.
4. Gives you **one comparison table** for the paper, plus the trained model files for the website.
5. *(Optional, Part B)* trains the **DNA-shape** model too. It needs the R tool DNAshapeR, which takes extra setup.

**Before you start**
- `Runtime → Change runtime type → T4 GPU`, then Save.
- Put **`tfbs_project.zip`** (the project code) in a Google Drive folder called **`TFBS`**: `My Drive/TFBS/tfbs_project.zip`.
- Run the cells from top to bottom. Results are saved to Drive after every finished run, so if Colab disconnects, reconnect and run the cells again. It continues where it stopped.

**Time:** the quick test takes about 2 minutes. The full run is roughly 1.5–3 hours on a free T4 GPU (3 models × 5 folds).

## 1. Check the GPU

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU! Go to Runtime > Change runtime type > T4 GPU, then run this cell again.")

## 2. Connect Google Drive and unpack the project code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, shutil
WORK = '/content/drive/MyDrive/TFBS'          # your Drive folder
PROJECT = '/content/tfbs_project'             # working copy inside Colab
os.makedirs(WORK, exist_ok=True)

zip_path = os.path.join(WORK, 'tfbs_project.zip')
assert os.path.exists(zip_path), f'Put tfbs_project.zip in {WORK} first.'
shutil.rmtree(PROJECT, ignore_errors=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall('/content')
print('Project unpacked:', sorted(os.listdir(PROJECT)))

## 3. Get the dataset (FOSL2, 101 bp)

If you've put your own `train.csv` and `test.csv` in `My Drive/TFBS/`, those are used. Otherwise the cell downloads them from the public BERT-TFBS repository. It's the same dataset Udit trained the original CNN on.

Each row is `chr_id, sequence, label`. `label = 1` means FOSL2 really binds there according to the ChIP-seq experiment. `label = 0` is a shuffled copy of a real site with the same letters in a different order (marked `_shuf`), so the model can't cheat by counting letters.

In [ ]:
import pandas as pd, urllib.request, hashlib
RAW = f'{PROJECT}/ml/data/raw'
os.makedirs(RAW, exist_ok=True)
BASE = 'https://raw.githubusercontent.com/ZX1998-12/BERT-TFBS/master/Dataset/wgEncodeAwgTfbsHaibHepg2Fosl2V0416101UniPk/'

for name in ('train.csv', 'test.csv'):
    mine = os.path.join(WORK, name)
    if os.path.exists(mine):
        shutil.copy(mine, f'{RAW}/{name}'); print(f'Using your {name} from Drive')
    else:
        urllib.request.urlretrieve(BASE + name, f'{RAW}/{name}'); print(f'Downloaded {name}')

for name in ('train.csv', 'test.csv'):
    df = pd.read_csv(f'{RAW}/{name}')
    print(f"{name}: {len(df)} rows | bound={int((df.label==1).sum())} not bound={int((df.label==0).sum())} | "
          f"lengths {df.sequence.str.len().min()}-{df.sequence.str.len().max()} bp")
    assert set(df.columns) >= {'chr_id', 'sequence', 'label'}, 'unexpected columns'

## 4. Clean the data (the team's existing `prepare_dataset.py`)

In [ ]:
%cd {PROJECT}/ml
!python prepare_dataset.py | tail -n 25

## 5. Quick test (about 2 minutes)

This trains every model for 1 round on a small slice, just to prove everything works before the long run. **The numbers here don't mean anything.** Only check that it ends with a table and no red error.

In [ ]:
!TORCH_NUM_THREADS=4 python train_all.py --quick --out "{WORK}/results"

## 6. Full training (1.5–3 hours)

Trains CNN, CNN + BiLSTM and CNN + BiLSTM + Transformer, 5 times each.
- Keep this browser tab open. If Colab disconnects, run cells 1–4 again, then this cell. Finished runs are skipped automatically.
- Progress is saved in `My Drive/TFBS/results/`.

In [ ]:
!TORCH_NUM_THREADS=4 python train_all.py --out "{WORK}/results"

## 7. Results table (copy this into the paper)

Each number is the **average over the 5 runs ± how much it varied** (standard deviation), measured on the untouched test set.
- **AUROC**: how well the model ranks bound above unbound sequences (1.0 = perfect, 0.5 = guessing). This is the main number.
- **PR-AUC**: similar, but focused on how reliable the positive predictions are.
- **Accuracy / Precision / Recall / F1**: measured at the cut-off chosen on the validation data.

In [ ]:
import json
from IPython.display import Markdown, display
display(Markdown(open(f'{WORK}/results/results_table.md').read()))

import matplotlib.pyplot as plt
summary = json.load(open(f'{WORK}/results/cv_results.json'))['summary']
names = [r['model'] for r in summary.values()]
means = [r['roc_auc']['mean'] for r in summary.values()]
stds = [r['roc_auc']['std'] for r in summary.values()]
plt.figure(figsize=(7, 3.5))
plt.bar(names, means, yerr=stds, capsize=6, color='#3b82f6')
plt.ylim(min(means) - 0.05, 1.0); plt.ylabel('Test AUROC (mean ± sd, 5 folds)')
plt.xticks(rotation=12); plt.title('FOSL2 (HepG2): model comparison'); plt.tight_layout()
plt.savefig(f'{WORK}/results/auroc_comparison.png', dpi=200); plt.show()

## 8. Download the trained models for the website

For each model, the run with the best **validation** score is kept. We never pick by test score, because that would be cheating. They're saved in `My Drive/TFBS/results/final_models/`.

**To use them in the website:** copy `tri_branch.pth` into the project's `ml/models/` folder and rename it to **`best_hybrid.pth`**, replacing the old file. The website loads it automatically.

In [ ]:
print(open(f'{WORK}/results/final_models/chosen_folds.json').read())
shutil.make_archive(f'{WORK}/trained_models', 'zip', f'{WORK}/results/final_models')
print('Zipped to My Drive/TFBS/trained_models.zip')

---
# Part B (optional): the DNA-shape model

**What DNA shape means:** the model in Part A only reads the *letters* of the DNA. But the letters also change the DNA's physical *shape*: how wide its groove is, and how twisted or tilted each step is. Some proteins recognise their spot partly by feeling that shape. **DNAshapeR** is a published tool (Zhou et al., 2013; Chiu et al., 2016) that looks up these shape values for every position from a table built from physics simulations. This part gives the main model those shape values as an extra input and checks whether it predicts better.

Whatever the result, it's a valid finding for the paper: "shape helped by X" or "shape didn't help for FOSL2".

**Extra time:** about 15–25 minutes to install R and DNAshapeR, a few minutes to compute shapes, then about 1 hour of training.

In [ ]:
# Install R and DNAshapeR (only needed once per Colab session)
!which Rscript || (apt-get -qq update && apt-get -qq install -y r-base > /dev/null)
!Rscript -e 'if (!requireNamespace("BiocManager", quietly=TRUE)) install.packages("BiocManager", repos="https://cloud.r-project.org"); BiocManager::install("DNAshapeR", ask=FALSE, update=FALSE)' > /tmp/r_install.log 2>&1; tail -n 3 /tmp/r_install.log
!Rscript -e 'library(DNAshapeR); cat("DNAshapeR", as.character(packageVersion("DNAshapeR")), "ready\n")'

In [ ]:
# Compute the shape values for every sequence (the team's prepare_shape_dataset.py)
!python prepare_shape_dataset.py --batch-size 5000 --cache-dir ""

In [ ]:
# Train the DNA-shape model with the same 5 folds, then show the updated table
!TORCH_NUM_THREADS=4 python train_all.py --models shape --out "{WORK}/results"
from IPython.display import Markdown, display
display(Markdown(open(f'{WORK}/results/results_table.md').read()))

**Website note:** don't copy `shape.pth` into the website unless every laptop that runs the website has R and DNAshapeR installed. The default website uses `best_hybrid.pth`, which needs no R.